## Część 0. GOTOWE: uruchom po kolei, bez zmian

Trzy komórki: importy i model, funkcje z wykładu, wczytanie zdjęcia i pięciu kadrów do ćwiczeń.

In [ ]:
# GOTOWE: uruchom bez zmian (Shift + Enter)
import functools
import time
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import torch
from affine import Affine
from huggingface_hub.errors import GatedRepoError
from IPython.display import display
from matplotlib.patches import Patch, Rectangle
from rasterio.features import rasterize, shapes
from shapely.geometry import shape
from shapely.ops import unary_union
from transformers import AutoModel, AutoProcessor

plt.rcParams.update({"figure.dpi": 90, "axes.titlesize": 11})

SCIEZKA_ZDJECIA = Path("dane/lomianki_ortofoto.tif")          # zdjęcie do zadań
KATALOG_WYNIKOW = Path("wyniki/zadania_sam3")                  # tu trafiają Twoje wyniki
KATALOG_WYNIKOW.mkdir(parents=True, exist_ok=True)
if not SCIEZKA_ZDJECIA.exists():
    raise FileNotFoundError("Brak pliku dane/lomianki_ortofoto.tif. Uruchom w terminalu:  python pobierz_lomianki.py")

# SAM3-LiteText: SAM 3 z lżejszym enkoderem tekstu, bez bramki (jak na wykładzie). Pełny SAM 3 to "facebook/sam3" (wymaga zgody Meta).
ID_MODELU = "vil-uob/sam3-litetext-s0"
URZADZENIE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

# Klasy: te same co w poziomie 3 (VDD). Piksel, o którym model nic nie powiedział, dostaje NIEZNANE (to NIE jest klasa „inne”).
NAZWY_KLAS = ["inne", "ściana", "droga", "roślinność", "pojazd", "dach", "woda"]
KOLORY_KLAS = ["#8c8c8c", "#e63c3c", "#ffffff", "#28a028", "#0078ff", "#ff9600", "#00dcdc"]
KOLORY_RGB = np.array([[int(k[i:i + 2], 16) for i in (1, 3, 5)] for k in KOLORY_KLAS], dtype=np.uint8)
N_KLAS = len(NAZWY_KLAS)
NIEZNANE = 255

print(f"Wczytuję model na urządzeniu „{URZADZENIE}” (ok. 10 s; za pierwszym razem dłużej, bo model się pobiera)...")
t0 = time.perf_counter()
try:
    procesor = AutoProcessor.from_pretrained(ID_MODELU)
    model = AutoModel.from_pretrained(ID_MODELU).to(URZADZENIE).eval()
except GatedRepoError as blad:
    raise RuntimeError(f"Brak dostępu do {ID_MODELU} (wagi „gated”). Zostaw ID_MODELU = \"vil-uob/sam3-litetext-s0\".") from blad
print(f"✅ Model gotowy ({time.perf_counter() - t0:.0f} s).")

In [ ]:
# GOTOWE: uruchom bez zmian (Shift + Enter)
# ===== Funkcje z wykładu (skopiowane z notebooka 04, bez zmian) =====
PROG = 0.5                        # minimalna pewność obiektu
KAFEL_M, ZAKLADKA_M = 60, 15      # rozmiar kafla i zakładka [m]: wartości domyślne funkcji poniżej
KLASY_DO_WYBORU = ["roślinność", "droga", "dach", "pojazd"]      # klasy, które umie ten notebook (od spodu do wierzchu)
KOLEJNOSC = KLASY_DO_WYBORU
POJECIE_NA_KLASE = {}             # słownik fraza → klasa podajemy w zadaniu 5
KOLUMNY_GDF = ["id", "pojecie", "klasa", "wynik", "pow_m2", "geometry"]

def wczytaj_zdjecie(sciezka, gsd_reczne=None):
    """Zwraca: rgb (H,W,3 uint8), waznie (H,W bool), profil (transform, crs), gsd [m/piksel]."""
    with rasterio.open(sciezka) as src:
        if src.count < 3 or src.dtypes[0] != "uint8":
            raise ValueError(f"Oczekuję 8-bitowego RGB, a jest: {src.count} pasm, {src.dtypes[0]}")
        rgb = np.moveaxis(src.read([1, 2, 3]), 0, -1)
        profil = {"transform": src.transform, "crs": src.crs}
        metryczny = src.crs is not None and src.crs.is_projected
        gsd = src.res[0] if metryczny else gsd_reczne
    waznie = ~(rgb == 0).all(axis=-1)          # czarne tło ortomozaiku = brak danych
    if gsd is None:
        raise ValueError("Zdjęcie nie jest w układzie metrycznym — podaj gsd_reczne (m/piksel).")
    return rgb, waznie, profil, gsd


def nakladka(rgb, maska, kolor=(255, 0, 255), alfa=0.55):
    """Półprzezroczysta maska nałożona na zdjęcie (do oceny wzrokowej)."""
    wynik = rgb.astype(np.float32)
    wynik[maska] = (1 - alfa) * wynik[maska] + alfa * np.array(kolor, dtype=np.float32)
    return wynik.astype(np.uint8)


def koloruj_obiekty(obraz, obiekty, alfa=0.6, seed=1):
    """Każdy obiekt w innym losowym kolorze. Większe maluje pierwsze, mniejsze na wierzchu — bo maski się nakładają."""
    rng = np.random.default_rng(seed)
    wynik = obraz.astype(np.float32)
    for o in sorted(obiekty, key=lambda o: -o["piksele"]):
        wys, szer = o["maska"].shape
        okno = wynik[o["wiersz"]:o["wiersz"] + wys, o["kolumna"]:o["kolumna"] + szer]
        okno[o["maska"]] = (1 - alfa) * okno[o["maska"]] + alfa * rng.integers(40, 255, 3)
    return wynik.astype(np.uint8)


def maska_z_obiektow(obiekty, ksztalt):
    """Suma (OR) masek obiektów jako jedna maska bool o rozmiarze `ksztalt` — nakładające się obiekty liczą się raz."""
    wynik = np.zeros(ksztalt, dtype=bool)
    for o in obiekty:
        wys, szer = o["maska"].shape
        wynik[o["wiersz"]:o["wiersz"] + wys, o["kolumna"]:o["kolumna"] + szer] |= o["maska"]
    return wynik


def koduj_obraz(obraz):
    """Ciężki krok: RGB uint8 (H,W,3) → cechy obrazu (do wielokrotnego użycia) i rozmiar oryginału [[H, W]]."""
    wejscie = procesor(images=obraz, return_tensors="pt")
    with torch.no_grad():
        cechy = model.get_vision_features(pixel_values=wejscie["pixel_values"].to(URZADZENIE))
    return cechy, wejscie["original_sizes"].tolist()


def znajdz_pojecie(cechy, rozmiar, pojecie, prog=PROG, wiersz0=0, kolumna0=0, ramki=(), etykiety_ramek=()):
    """Lekki krok: fraza -> obiekty. `ramki` = [(x1, y1, x2, y2), ...] w pikselach obrazu podanego do `koduj_obraz`,
    `etykiety_ramek`: 1 = „takie też chcę”, 0 = „takich nie chcę”. (wiersz0, kolumna0) = położenie obrazu w całym zdjęciu."""
    if len(ramki):
        wejscie = procesor(text=pojecie, input_boxes=[[[float(v) for v in r] for r in ramki]],
                           input_boxes_labels=[[int(e) for e in etykiety_ramek]], original_sizes=rozmiar, return_tensors="pt")
    else:
        wejscie = procesor(text=pojecie, return_tensors="pt")
    argumenty = {k: v.to(URZADZENIE) for k, v in wejscie.items()
                 if k in ("input_ids", "attention_mask", "input_boxes", "input_boxes_labels")}
    with torch.no_grad():
        wyjscie = model(vision_embeds=cechy, **argumenty)
    wynik = procesor.post_process_instance_segmentation(wyjscie, threshold=prog, mask_threshold=0.5, target_sizes=rozmiar)[0]

    obiekty = []
    for maska, pewnosc in zip(wynik["masks"].cpu().numpy().astype(bool), wynik["scores"].cpu().numpy()):
        wiersze, kolumny = np.flatnonzero(maska.any(1)), np.flatnonzero(maska.any(0))
        if len(wiersze) == 0:
            continue
        w1, w2, k1, k2 = wiersze[0], wiersze[-1] + 1, kolumny[0], kolumny[-1] + 1
        obiekty.append({"maska": maska[w1:w2, k1:k2].copy(), "wiersz": wiersz0 + int(w1), "kolumna": kolumna0 + int(k1),
                        "piksele": int(maska.sum()), "wynik": float(pewnosc), "pojecie": pojecie})
    return obiekty


def obecnosc_frazy(cechy, fraza):
    """Czy fraza w ogóle występuje na obrazie według modelu: `obecność` (0–1, ocena całego obrazu), najwyższa pewność
    pojedynczego obiektu i liczba obiektów przy progu PROG."""
    wejscie = procesor(text=fraza, return_tensors="pt")
    with torch.no_grad():
        wyjscie = model(vision_embeds=cechy, input_ids=wejscie["input_ids"].to(URZADZENIE),
                        attention_mask=wejscie["attention_mask"].to(URZADZENIE))
    pewnosci = (wyjscie.pred_logits.sigmoid() * wyjscie.presence_logits.sigmoid()).flatten()
    return {"obecność": round(wyjscie.presence_logits.sigmoid().item(), 2), "najwyższa pewność": round(pewnosci.max().item(), 2),
            f"obiektów (≥ {PROG})": int((pewnosci >= PROG).sum())}


def poczatki_kafli(n, kafel, zakladka):
    """Początki kafli o boku `kafel` px, równomiernie pokrywających `n` px, z zakładką nie mniejszą niż `zakladka`."""
    if n <= kafel:
        return [0]
    liczba = int(np.ceil((n - zakladka) / (kafel - zakladka)))
    return np.linspace(0, n - kafel, liczba).round().astype(int).tolist()


def iou_obiektow(a, b):
    """IoU dwóch obiektów zapisanych jako przycięte maski z położeniem (bez budowania pełnowymiarowych masek)."""
    w0, k0 = max(a["wiersz"], b["wiersz"]), max(a["kolumna"], b["kolumna"])
    w1 = min(a["wiersz"] + a["maska"].shape[0], b["wiersz"] + b["maska"].shape[0])
    k1 = min(a["kolumna"] + a["maska"].shape[1], b["kolumna"] + b["maska"].shape[1])
    if w1 <= w0 or k1 <= k0:
        return 0.0
    ma = a["maska"][w0 - a["wiersz"]:w1 - a["wiersz"], k0 - a["kolumna"]:k1 - a["kolumna"]]
    mb = b["maska"][w0 - b["wiersz"]:w1 - b["wiersz"], k0 - b["kolumna"]:k1 - b["kolumna"]]
    wspolna = (ma & mb).sum()
    return wspolna / (a["piksele"] + b["piksele"] - wspolna)


def czesc_wazna(obiekt, waznie):
    """Jaka część pikseli obiektu leży na ważnych danych (a nie na czarnym tle ortomozaiku)."""
    wys, szer = obiekt["maska"].shape
    return waznie[obiekt["wiersz"]:obiekt["wiersz"] + wys, obiekt["kolumna"]:obiekt["kolumna"] + szer][obiekt["maska"]].mean()


def maski_tekstem_kafelkami(rgb, waznie, pojecia, gsd, kafel_m=KAFEL_M, zakladka_m=ZAKLADKA_M, prog=PROG, min_obiekt_m2=1.0):
    """SAM 3 na kafelkach z zakładką, dla każdej frazy. Zwraca {fraza: obiekty w układzie całego zdjęcia}."""
    wys_zdj, szer_zdj = waznie.shape
    kafel, zakladka = round(kafel_m / gsd), round(zakladka_m / gsd)
    kandydaci = {p: [] for p in pojecia}
    for w0 in poczatki_kafli(wys_zdj, kafel, zakladka):
        for k0 in poczatki_kafli(szer_zdj, kafel, zakladka):
            okno = np.s_[w0:w0 + kafel, k0:k0 + kafel]
            if waznie[okno].mean() < 0.05:                       # kafel prawie bez danych
                continue
            wys, szer = waznie[okno].shape
            cechy, rozmiar = koduj_obraz(rgb[okno].copy())       # koder obrazu raz na kafel, potem fraza po frazie
            for p in pojecia:
                for o in znajdz_pojecie(cechy, rozmiar, p, prog, w0, k0):
                    h, w = o["maska"].shape
                    y, x = o["wiersz"] - w0, o["kolumna"] - k0        # położenie maski w kaflu
                    ucieta = ((x == 0 and k0 > 0) or (y == 0 and w0 > 0)                       # dotyka wewnętrznej krawędzi kafla
                              or (x + w == szer and k0 + szer < szer_zdj) or (y + h == wys and w0 + wys < wys_zdj))
                    if not ucieta and czesc_wazna(o, waznie) >= 0.5 and o["piksele"] * gsd ** 2 >= min_obiekt_m2:
                        kandydaci[p].append(o)

    wynik = {}
    for p, lista in kandydaci.items():
        lista.sort(key=lambda o: -o["wynik"])                    # najpewniejsze pierwsze
        zostawione = []
        for o in lista:
            if all(iou_obiektow(o, z) < 0.7 for z in zostawione):    # duplikat z zakładki?
                zostawione.append(o)
        wynik[p] = zostawione
    return wynik


def maska_klas(obiekty, ksztalt, waznie, pojecie_na_klase=POJECIE_NA_KLASE, kolejnosc=KOLEJNOSC):
    """{fraza: obiekty} -> maska klas (uint8; NIEZNANE tam, gdzie nic nie wskazano). Zwraca też liczbę pikseli spornych
    (zgłoszonych przez więcej niż jedną klasę)."""
    klasy = np.full(ksztalt, NIEZNANE, dtype=np.uint8)
    zgloszenia = np.zeros(ksztalt, dtype=np.uint8)
    for nazwa in kolejnosc:
        pokrycie = np.zeros(ksztalt, dtype=bool)
        for fraza, klasa in pojecie_na_klase.items():
            if klasa == nazwa:
                pokrycie |= maska_z_obiektow(obiekty.get(fraza, []), ksztalt)
        klasy[pokrycie] = NAZWY_KLAS.index(nazwa)
        zgloszenia += pokrycie
    klasy[~waznie] = NIEZNANE
    return klasy, int((zgloszenia[waznie] > 1).sum())


def podglad_klas(rgb, klasy, alfa=0.55):
    """Zdjęcie z maską klas; piksele „nieznane” zostają nietknięte."""
    wynik = rgb.astype(np.float32)
    znane = klasy != NIEZNANE
    wynik[znane] = (1 - alfa) * wynik[znane] + alfa * KOLORY_RGB[klasy[znane]]
    return wynik.astype(np.uint8)


def legenda(klasy_uzyte):
    return [Patch(facecolor=KOLORY_KLAS[i], edgecolor="0.4", label=NAZWY_KLAS[i]) for i in sorted(klasy_uzyte)]


def zapisz_maske_klas(klasy, profil, sciezka):
    """Maska klas (uint8, 255 = nieznane) jako GeoTIFF na siatce zdjęcia, z paletą kolorów poziomu 3."""
    with rasterio.open(sciezka, "w", driver="GTiff", height=klasy.shape[0], width=klasy.shape[1], count=1, dtype="uint8",
                       crs=profil["crs"], transform=profil["transform"], nodata=NIEZNANE, compress="deflate") as dst:
        dst.write(klasy, 1)
        dst.write_colormap(1, {i: (*map(int, kolor), 255) for i, kolor in enumerate(KOLORY_RGB)})


def maska_do_poligonu(maska, transform, gsd):
    """Maska (bool) -> jeden poligon (lub multipoligon) w układzie mapy, uproszczony o 1 piksel.
    `transform` musi opisywać położenie samej tablicy `maska` (nie całego zdjęcia)."""
    czesci = [shape(g) for g, _ in shapes(maska.astype(np.uint8), mask=maska, transform=transform)]
    return unary_union(czesci).simplify(gsd)


def obiekty_do_gdf(obiekty, profil, gsd, pojecie_na_klase=POJECIE_NA_KLASE):
    """{fraza: obiekty} -> GeoDataFrame: poligon w układzie zdjęcia + fraza, klasa, pewność, powierzchnia."""
    wiersze = []
    for fraza, lista in obiekty.items():
        for o in lista:
            wiersze.append({
                "id": len(wiersze) + 1, "pojecie": fraza, "klasa": pojecie_na_klase[fraza], "wynik": round(o["wynik"], 2),
                "pow_m2": round(o["piksele"] * gsd ** 2, 1),
                "geometry": maska_do_poligonu(o["maska"], profil["transform"] * Affine.translation(o["kolumna"], o["wiersz"]), gsd)})
    return gpd.GeoDataFrame(pd.DataFrame(wiersze, columns=KOLUMNY_GDF), geometry="geometry", crs=profil["crs"])


def rasteryzuj_gdf(gdf, ksztalt, transform, waznie, kolejnosc=KOLEJNOSC):
    """Poligony z kolumną `klasa` -> maska klas (ta sama kolejność malowania: od spodu do wierzchu)."""
    klasy = np.full(ksztalt, NIEZNANE, dtype=np.uint8)
    kolejnosc_klas = {nazwa: i for i, nazwa in enumerate(kolejnosc)}
    gdf = gdf.assign(_k=gdf["klasa"].map(kolejnosc_klas)).sort_values("_k")
    ksztalty = [(g, NAZWY_KLAS.index(k)) for g, k in zip(gdf.geometry, gdf["klasa"])]
    if ksztalty:
        rasterize(ksztalty, out=klasy, transform=transform)
    klasy[~waznie] = NIEZNANE
    return klasy


# ===== Opakowania na potrzeby zadań: sprawdzają to, co wpisałeś, i rysują wynik =====
class BladUzytkownika(ValueError):
    """Błąd w ustawieniach wpisanych przez uczestnika: pokazujemy zrozumiały komunikat zamiast długiego raportu błędu."""


def przyjazne(funkcja):
    @functools.wraps(funkcja)
    def opakowana(*argumenty, **nazwane):
        try:
            return funkcja(*argumenty, **nazwane)
        except BladUzytkownika as blad:
            print(f"⚠️  {blad}")
    return opakowana


def sprawdz_kadr(kadr):
    kadr = str(kadr).strip().upper()
    if kadr not in KADRY:
        raise BladUzytkownika(f"Nie ma kadru „{kadr}”. Wpisz jedną z liter: {', '.join(KADRY)} (w cudzysłowie, np. \"B\").")
    return kadr


def sprawdz_fraze(fraza):
    fraza = str(fraza).strip().lower()
    if not fraza:
        raise BladUzytkownika("Fraza jest pusta. Wpisz krótką nazwę po angielsku, np. \"bush\" albo \"dirt road\".")
    if not fraza.isascii():
        raise BladUzytkownika(f"Fraza „{fraza}” ma polskie litery. Model rozumie krótkie frazy po angielsku, np. \"bush\", \"dirt road\".")
    if len(fraza.split()) > 4:
        raise BladUzytkownika(f"Fraza „{fraza}” jest za długa. Użyj jednego do trzech słów.")
    return fraza


def sprawdz_prog(prog, nazwa="PROG"):
    try:
        prog = float(prog)
    except (TypeError, ValueError):
        raise BladUzytkownika(f"{nazwa} musi być liczbą od 0.05 do 0.95, np. 0.5 (z KROPKĄ, nie z przecinkiem).") from None
    if not 0.05 <= prog <= 0.95:
        raise BladUzytkownika(f"{nazwa} = {prog} jest poza zakresem. Wpisz liczbę od 0.05 do 0.95.")
    return round(prog, 2)


def pobierz_kadr(kadr):
    wiersz0, kolumna0, _ = KADRY[kadr]
    return np.ascontiguousarray(rgb[wiersz0:wiersz0 + ROZMIAR_KADRU, kolumna0:kolumna0 + ROZMIAR_KADRU])


_CECHY_KADROW = {}


def cechy_kadru(kadr):
    """Koder obrazu (ciężki krok) liczy się raz na kadr; kolejne frazy na tym samym kadrze są szybkie."""
    if kadr not in _CECHY_KADROW:
        _CECHY_KADROW[kadr] = koduj_obraz(pobierz_kadr(kadr))
    return _CECHY_KADROW[kadr]


def opis_kadru(kadr):
    return f"Kadr {kadr}: {KADRY[kadr][2]}"


@przyjazne
def zadanie_obecnosc(kadr, frazy):
    """Zadanie 1: dla każdej frazy „obecność” na kadrze (0 = model tego nie widzi, 1 = na pewno jest)."""
    kadr = sprawdz_kadr(kadr)
    if isinstance(frazy, str) or not len(frazy):
        raise BladUzytkownika("FRAZY to lista w nawiasach kwadratowych, np. [\"tree\", \"bush\"]. Frazy oddziel przecinkami.")
    frazy = [sprawdz_fraze(f) for f in frazy]
    cechy, _ = cechy_kadru(kadr)
    wiersze = []
    for f in frazy:
        o = obecnosc_frazy(cechy, f)
        wiersze.append({"fraza": f, "obecność": o["obecność"], "najwyższa pewność": o["najwyższa pewność"],
                        f"obiektów (pewność ≥ {PROG})": o[f"obiektów (≥ {PROG})"]})
    tabela = pd.DataFrame(wiersze).set_index("fraza")
    tabela["werdykt"] = np.where(tabela["obecność"] >= 0.5, "model to widzi",
                                 np.where(tabela["obecność"] >= 0.3, "niepewnie", "model tego nie widzi"))
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.6), gridspec_kw={"width_ratios": [1, 1.4]})
    ax[0].imshow(pobierz_kadr(kadr)); ax[0].set_title(opis_kadru(kadr)); ax[0].axis("off")
    ax[1].barh(range(len(tabela)), tabela["obecność"], color=["#28a028" if v >= 0.5 else "#b0b0b0" for v in tabela["obecność"]])
    ax[1].set_yticks(range(len(tabela)), tabela.index); ax[1].invert_yaxis(); ax[1].set_xlim(0, 1)
    ax[1].axvline(0.5, color="k", ls="--", lw=1)
    ax[1].set_xlabel("obecność frazy na kadrze: 0 = brak, 1 = na pewno jest (kreska = 0.5)")
    ax[1].set_title("Czy model widzi to na zdjęciu?")
    plt.tight_layout(); plt.show()
    display(tabela)


DZIENNIK_FRAZ = []


@przyjazne
def zadanie_fraza(kadr, fraza, prog=PROG, pokaz_dziennik=True):
    """Zadanie 2: fraza → maski obiektów na kadrze; każda próba ląduje w dzienniku pod rysunkiem."""
    kadr, fraza, prog = sprawdz_kadr(kadr), sprawdz_fraze(fraza), sprawdz_prog(prog)
    obraz = pobierz_kadr(kadr)
    cechy, rozmiar = cechy_kadru(kadr)
    obiekty = znajdz_pojecie(cechy, rozmiar, fraza, prog)
    pow_m2 = maska_z_obiektow(obiekty, obraz.shape[:2]).sum() * gsd ** 2
    fig, ax = plt.subplots(1, 2, figsize=(11, 5.4))
    ax[0].imshow(obraz); ax[0].set_title(opis_kadru(kadr))
    ax[1].imshow(koloruj_obiekty(obraz, obiekty)); ax[1].set_title(f"„{fraza}”: {len(obiekty)} obiekt., {pow_m2:.0f} m²")
    for a in ax:
        a.axis("off")
    plt.tight_layout(); plt.show()
    DZIENNIK_FRAZ.append({"kadr": kadr, "fraza": fraza, "próg": prog, "obiektów": len(obiekty), "powierzchnia [m²]": round(pow_m2),
                          "najwyższa pewność": round(max((o["wynik"] for o in obiekty), default=0), 2)})
    if not obiekty:
        print("Nic nie znaleziono. Albo tego nie ma na kadrze, albo model zna to pod inną nazwą: sprawdź „obecność” (zadanie 1) i spróbuj innej frazy.")
    if pokaz_dziennik:
        print("Dziennik Twoich prób (ostatnie 10):")
        display(pd.DataFrame(DZIENNIK_FRAZ).tail(10))


@przyjazne
def zadanie_prog(kadr, fraza, progi):
    """Zadanie 3: ta sama fraza przy różnych progach pewności (model liczy się raz, resztę robi odsiew)."""
    kadr, fraza = sprawdz_kadr(kadr), sprawdz_fraze(fraza)
    if isinstance(progi, (int, float, str)) or len(progi) != 3:
        raise BladUzytkownika("PROGI to trzy liczby w nawiasie okrągłym, np. (0.1, 0.3, 0.5).")
    progi = [sprawdz_prog(p, "PROGI") for p in progi]
    obraz = pobierz_kadr(kadr)
    cechy, rozmiar = cechy_kadru(kadr)
    luzne = znajdz_pojecie(cechy, rozmiar, fraza, prog=0.05)
    liczba = lambda p: sum(o["wynik"] >= p for o in luzne)
    powierzchnia = lambda p: maska_z_obiektow([o for o in luzne if o["wynik"] >= p], obraz.shape[:2]).sum() * gsd ** 2
    skala = np.round(np.arange(0.1, 0.96, 0.05), 2)
    fig, ax = plt.subplots(1, 4, figsize=(17, 4.4), gridspec_kw={"width_ratios": [1.5, 1, 1, 1]})
    ax[0].plot(skala, [liczba(p) for p in skala], "o-", ms=3, color="C0")
    ax[0].set_xlabel("próg pewności"); ax[0].set_ylabel("liczba obiektów")
    for p in progi:
        ax[0].axvline(p, color="C3", ls=":", lw=1)
    ax[0].set_title(f"{opis_kadru(kadr)}, „{fraza}”: ile obiektów zostaje", fontsize=9)
    for a, p in zip(ax[1:], progi):
        zostaje = [o for o in luzne if o["wynik"] >= p]
        a.imshow(nakladka(obraz, maska_z_obiektow(zostaje, obraz.shape[:2])))
        a.set_title(f"próg {p}: {len(zostaje)} obiekt., {powierzchnia(p):.0f} m²"); a.axis("off")
    plt.tight_layout(); plt.show()
    wybrane = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
    display(pd.DataFrame({"obiektów": [liczba(p) for p in wybrane], "powierzchnia [m²]": [round(powierzchnia(p)) for p in wybrane]},
                         index=wybrane).rename_axis("próg").T)


@przyjazne
def zadanie_ramka(kadr, fraza, ramka, prog=PROG):
    """Zadanie 4: fraza bez ramki i z ramką wzorcową („szukaj czegoś takiego jak to w ramce”)."""
    kadr, fraza, prog = sprawdz_kadr(kadr), sprawdz_fraze(fraza), sprawdz_prog(prog)
    try:
        x1, y1, x2, y2 = [float(v) for v in ramka]
    except (TypeError, ValueError):
        raise BladUzytkownika("RAMKA to cztery liczby w nawiasie: (x lewy, y górny, x prawy, y dolny), np. (225, 115, 315, 225).") from None
    obraz = pobierz_kadr(kadr)
    wys, szer = obraz.shape[:2]
    if not (0 <= x1 < x2 <= szer and 0 <= y1 < y2 <= wys):
        raise BladUzytkownika(f"Ramka musi mieścić się w kadrze ({szer} × {wys} px): lewy < prawy, górny < dolny, liczby od 0 do {szer}.")
    cechy, rozmiar = cechy_kadru(kadr)
    bez_ramki = znajdz_pojecie(cechy, rozmiar, fraza, prog)
    z_ramka = znajdz_pojecie(cechy, rozmiar, fraza, prog, ramki=[(x1, y1, x2, y2)], etykiety_ramek=[1])
    fig, ax = plt.subplots(1, 3, figsize=(17, 5.6))
    ax[0].imshow(obraz)
    ax[0].add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, ec="yellow", lw=2.5))
    ax[0].set_xticks(range(0, szer + 1, 50)); ax[0].set_yticks(range(0, wys + 1, 50))
    ax[0].grid(color="yellow", alpha=0.4, lw=0.6); ax[0].tick_params(labelsize=8)
    ax[0].set_title(f"{opis_kadru(kadr)}: Twoja ramka (żółta); osie = piksele")
    for a, (opis, obiekty) in zip(ax[1:], [(f"sama fraza „{fraza}”", bez_ramki), (f"fraza „{fraza}” + ramka", z_ramka)]):
        a.imshow(koloruj_obiekty(obraz, obiekty))
        a.set_title(f"{opis}: {len(obiekty)} obiekt., {maska_z_obiektow(obiekty, obraz.shape[:2]).sum() * gsd ** 2:.0f} m²"); a.axis("off")
    plt.tight_layout(); plt.show()
    print(f"Bez ramki: {len(bez_ramki)} obiekt.   |   z ramką: {len(z_ramka)} obiekt.")
    if z_ramka:
        display(pd.DataFrame([{"pewność": round(o["wynik"], 2), "powierzchnia [m²]": round(o["piksele"] * gsd ** 2),
                               "lewy górny róg (x, y)": (o["kolumna"], o["wiersz"])} for o in z_ramka],
                             index=range(1, len(z_ramka) + 1)).rename_axis("obiekt z ramką"))


DZIENNIK_KAFLI = []


@przyjazne
def zadanie_cala(frazy_i_klasy, kafel_m, prog=PROG):
    """Zadanie 5: całe zdjęcie na kafelkach → maska klas (GeoTIFF) i poligony (GPKG) do poprawy w QGIS."""
    if not isinstance(frazy_i_klasy, dict) or not frazy_i_klasy:
        raise BladUzytkownika("FRAZY_I_KLASY to słownik w klamrach: {\"fraza\": \"klasa\", ...}, np. {\"bush\": \"roślinność\"}.")
    frazy_i_klasy = {sprawdz_fraze(f): k for f, k in frazy_i_klasy.items()}
    for f, k in frazy_i_klasy.items():
        if k not in KLASY_DO_WYBORU:
            raise BladUzytkownika(f"Klasa „{k}” (przy frazie „{f}”) nie istnieje. Do wyboru: {', '.join(KLASY_DO_WYBORU)}.")
    try:
        kafel_m = float(kafel_m)
    except (TypeError, ValueError):
        kafel_m = 0
    if not 20 <= kafel_m <= 300:
        raise BladUzytkownika("KAFEL_M to liczba metrów od 20 do 300, np. 60 albo 120.")
    prog = sprawdz_prog(prog)
    kolejnosc = [k for k in KLASY_DO_WYBORU if k in frazy_i_klasy.values()]

    t0 = time.perf_counter()
    obiekty = maski_tekstem_kafelkami(rgb, waznie, list(frazy_i_klasy), gsd, kafel_m=kafel_m, zakladka_m=kafel_m / 4, prog=prog)
    czas = time.perf_counter() - t0
    klasy, _ = maska_klas(obiekty, waznie.shape, waznie, frazy_i_klasy, kolejnosc)
    liczniki = np.bincount(klasy[klasy != NIEZNANE], minlength=N_KLAS)
    oznaczone = liczniki.sum() / waznie.sum() * 100

    nazwa = SCIEZKA_ZDJECIA.stem
    tif, gpkg = KATALOG_WYNIKOW / f"{nazwa}_sam3_klasy.tif", KATALOG_WYNIKOW / f"{nazwa}_sam3_obiekty.gpkg"
    kopia = KATALOG_WYNIKOW / f"{nazwa}_sam3_obiekty_oryginal.gpkg"
    gdf = obiekty_do_gdf(obiekty, profil, gsd, frazy_i_klasy)
    KATALOG_WYNIKOW.mkdir(parents=True, exist_ok=True)                    # na wypadek, gdyby katalog został skasowany
    try:
        zapisz_maske_klas(klasy, profil, tif)
        plt.imsave(KATALOG_WYNIKOW / f"{nazwa}_sam3_podglad.png", podglad_klas(rgb, klasy))
        for plik in (gpkg, kopia):
            plik.unlink(missing_ok=True)
            if len(gdf):
                gdf.to_file(plik, layer="obiekty", driver="GPKG")
    except PermissionError:
        raise BladUzytkownika(f"Nie mogę nadpisać plików w {KATALOG_WYNIKOW}. Zamknij je w QGIS i uruchom komórkę jeszcze raz.") from None

    fig, ax = plt.subplots(1, 2, figsize=(14, 6.8))
    ax[0].imshow(rgb); ax[0].set_title("Zdjęcie")
    ax[1].imshow(podglad_klas(rgb, klasy)); ax[1].set_title(f"Maska klas, kafel {kafel_m:g} m (bez nakładki = „nieznane”)")
    if liczniki.sum():
        ax[1].legend(handles=legenda(np.flatnonzero(liczniki)), loc="lower left", fontsize=9)
    for a in ax:
        a.axis("off")
    plt.tight_layout(); plt.show()

    print(f"Kafel {kafel_m:g} m, czas {czas:.0f} s. Oznaczone: {oznaczone:.1f} % pikseli, reszta „nieznane” (pytaliśmy tylko o wpisane frazy).")
    if not len(gdf):
        print("Nic nie znaleziono: nie zapisano poligonów. Zmień frazy albo kafel.")
    else:
        print(f"Zapisano: {tif.name} (maska klas) i {gpkg.name} ({len(gdf)} poligonów) w {KATALOG_WYNIKOW}/")
    wiersze = []
    for fraza, klasa in frazy_i_klasy.items():
        lista = obiekty[fraza]
        wiersze.append({"fraza": fraza, "klasa": klasa, "obiektów": len(lista),
                        "powierzchnia [m²]": round(maska_z_obiektow(lista, waznie.shape).sum() * gsd ** 2),
                        "mediana pewności": round(float(np.median([o["wynik"] for o in lista])), 2) if lista else None})
    display(pd.DataFrame(wiersze).set_index("fraza"))
    DZIENNIK_KAFLI.append({"kafel [m]": kafel_m, "frazy": ", ".join(frazy_i_klasy), "obiektów razem": len(gdf),
                           "oznaczone [%]": round(oznaczone, 1), "czas [s]": round(czas)})
    print("Dziennik Twoich przebiegów (porównaj kafle):")
    display(pd.DataFrame(DZIENNIK_KAFLI).tail(6))


@przyjazne
def zadanie_po_poprawkach():
    """Zadanie 6: wczytuje poligony poprawione w QGIS, przelicza maskę klas i pokazuje, ile zmieniłeś."""
    nazwa = SCIEZKA_ZDJECIA.stem
    gpkg, kopia = KATALOG_WYNIKOW / f"{nazwa}_sam3_obiekty.gpkg", KATALOG_WYNIKOW / f"{nazwa}_sam3_obiekty_oryginal.gpkg"
    if not gpkg.exists() or not kopia.exists():
        raise BladUzytkownika("Nie ma jeszcze plików z zadania 5 (albo nic tam nie znaleziono). Uruchom najpierw zadanie 5.")
    przed, po = gpd.read_file(kopia, layer="obiekty"), gpd.read_file(gpkg, layer="obiekty")
    for tabela in (przed, po):
        tabela["id"] = pd.to_numeric(tabela["id"], errors="coerce")       # id bywa zapisany jako tekst albo z pustymi wartościami
    po = po[po.geometry.notna() & ~po.geometry.is_empty]                  # QGIS potrafi zostawić pusty obiekt
    usuniete = przed[~przed["id"].isin(po["id"])]
    nowe = po[~po["id"].isin(przed["id"])]                                # dorysowane w QGIS nie mają starego id
    pierwotne = dict(zip(przed["id"], przed.geometry))
    zmienione = sum(not g.equals_exact(pierwotne[i], 1e-6) for i, g in zip(po["id"], po.geometry) if i in pierwotne)
    if not len(usuniete) and not len(nowe) and not zmienione:
        raise BladUzytkownika("Nie widzę żadnych zmian w pliku. Zapisz zmiany w QGIS (przycisk „Zapisz zmiany warstwy”) i uruchom jeszcze raz.")
    zle = po[~po["klasa"].isin(KLASY_DO_WYBORU)]
    if len(zle):
        print(f"⚠️  {len(zle)} obiekt. nie ma poprawnej klasy w kolumnie „klasa” (wpisz: {', '.join(KLASY_DO_WYBORU)}). Pomijam je.")
    po = po[po["klasa"].isin(KLASY_DO_WYBORU)]
    stan_przed = rasteryzuj_gdf(przed, waznie.shape, profil["transform"], waznie, KLASY_DO_WYBORU)
    stan_po = rasteryzuj_gdf(po, waznie.shape, profil["transform"], waznie, KLASY_DO_WYBORU)
    zapisz_maske_klas(stan_po, profil, KATALOG_WYNIKOW / f"{nazwa}_sam3_klasy_poprawione.tif")

    oznaczone = ((stan_przed != NIEZNANE) | (stan_po != NIEZNANE)).sum()
    print(f"Z {len(przed)} obiektów z modelu: usunięto {len(usuniete)} ({len(usuniete) / len(przed):.0%}), zmieniono kształt {zmienione}, "
          f"dorysowano {len(nowe)}.")
    print(f"Zmieniło się {(stan_przed != stan_po).sum()} pikseli = {(stan_przed != stan_po).sum() / max(oznaczone, 1):.1%} oznaczonych. "
          f"Zapisano {nazwa}_sam3_klasy_poprawione.tif")
    fig, ax = plt.subplots(1, 2, figsize=(14, 6.8))
    for a, (opis, stan) in zip(ax, [("przed poprawkami (z modelu)", stan_przed), ("po Twoich poprawkach", stan_po)]):
        a.imshow(podglad_klas(rgb, stan)); a.set_title(opis); a.axis("off")
    plt.tight_layout(); plt.show()
    hektary = lambda stan: {k: round((stan == NAZWY_KLAS.index(k)).sum() * gsd ** 2 / 1e4, 3) for k in KLASY_DO_WYBORU}
    display(pd.DataFrame({"przed [ha]": hektary(stan_przed), "po [ha]": hektary(stan_po)}).T)

In [ ]:
# GOTOWE: uruchom bez zmian (Shift + Enter)
rgb, waznie, profil, gsd = wczytaj_zdjecie(SCIEZKA_ZDJECIA)

# Pięć kadrów do ćwiczeń: (wiersz, kolumna lewego górnego rogu, opis). Każdy ma 400 × 400 px (40 × 40 m).
ROZMIAR_KADRU = 400
KADRY = {
    "A": (750, 1000, "kępy krzewów i drzew"),
    "B": (800, 0, "droga gruntowa"),
    "C": (0, 0, "ściernisko i ścieżka w rogu"),
    "D": (1250, 550, "kępa krzewów z nawłocią"),
    "E": (1000, 1300, "pas nawłoci"),
}
for nazwa, (w0, k0, _) in KADRY.items():
    assert w0 + ROZMIAR_KADRU <= rgb.shape[0] and k0 + ROZMIAR_KADRU <= rgb.shape[1], f"Kadr {nazwa} wychodzi poza zdjęcie"

print(f"Zdjęcie: {rgb.shape[1]} × {rgb.shape[0]} px, piksel {gsd * 100:.0f} cm, układ {profil['crs']}, "
      f"{waznie.mean():.0%} pikseli ma dane (reszta to czarne tło ortomozaiku).")

fig, ax = plt.subplots(figsize=(7.5, 7.5))
ax.imshow(rgb)
for nazwa, (w0, k0, _) in KADRY.items():
    ax.add_patch(Rectangle((k0, w0), ROZMIAR_KADRU, ROZMIAR_KADRU, fill=False, ec="cyan", lw=2))
    ax.text(k0 + 12, w0 + 12, nazwa, color="cyan", fontsize=16, fontweight="bold", va="top")
ax.set_title("Całe zdjęcie z Łomianek i pięć kadrów do ćwiczeń"); ax.axis("off")
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(1, len(KADRY), figsize=(4 * len(KADRY), 4.3))
for a, nazwa in zip(ax, KADRY):
    a.imshow(pobierz_kadr(nazwa)); a.set_title(opis_kadru(nazwa), fontsize=10); a.axis("off")
plt.tight_layout(); plt.show()

## Krok 1. Nazwa

### Zadanie 1. Czy model widzi to, czego szukam?

Zanim poprosisz model o maski, **sprawdź, czy w ogóle rozpoznaje daną nazwę na tym zdjęciu**. Model ocenia każdą frazę liczbą
**obecność** od 0 do 1: blisko 1 = „to jest na zdjęciu”, blisko 0 = „tego nie widzę” (slajdy 6 i 19). Frazy piszesz **po angielsku**, krótko.

1. Uruchom komórkę **TWOJA KOLEJ** bez zmian (kadr A: kępy krzewów i drzew).
2. Zmień `KADR` na `"B"` (droga gruntowa) i uruchom jeszcze raz.
3. Wróć do kadru A i dopisz **dwie własne frazy** (pomysły: `"grass"`, `"path"`, `"tire tracks"`, `"vegetation"`, `"yellow plants"`).



In [ ]:
# ==================== ZMIEŃ TU ====================
KADR = "A"                                                                    # litera kadru: A, B, C, D albo E
FRAZY = ["tree", "bush", "shrub", "dirt road", "roof", "car", "goldenrod"]    # frazy po angielsku, oddzielone przecinkami
# ===================================================

zadanie_obecnosc(KADR, FRAZY)

---

## Krok 2. Maski na kadrze

### Zadanie 2. Fraza → maski: znajdź nazwę dla drogi gruntowej

Na kadrze B biegnie droga gruntowa (dwie koleiny). Sprawdź, **jak ją nazwać**, żeby model narysował ją poprawnie (slajdy 8 i 9: „jak sformułować frazę”).
Każda Twoja próba zapisuje się w **dzienniku** pod rysunkiem, więc możesz je porównać.

1. Uruchom komórkę **TWOJA KOLEJ** bez zmian (`"dirt road"`) i oceń wynik według trzech pytań (tak samo robi się to w sekcji 3 notebooka 04): czy znalezione obiekty to **naprawdę to**, czego szukam? czy **czegoś brakuje**? czy maska pokrywa **cały** obiekt?
2. Zmieniaj `FRAZA` i uruchamiaj za każdym razem: `"road"`, `"path"`, `"tire tracks"`, `"track"`. Wymyśl jedną własną.


In [ ]:
# ==================== ZMIEŃ TU ====================
KADR = "B"
FRAZA = "dirt road"          # spróbuj też: "road", "path", "tire tracks", "track"
# ===================================================

zadanie_fraza(KADR, FRAZA)

---

## Krok 3. Próg

### Zadanie 3. Próg pewności

Każdy znaleziony obiekt ma **pewność** (0–1). **Próg** to granica: obiekty poniżej niej odrzucamy (slajd 10). Za niski próg wpuszcza fałszywe alarmy i drobne fragmenty,
za wysoki gubi prawdziwe obiekty. Komórka pokazuje trzy progi obok siebie, wykres liczby obiektów i tabelę dla progów od 0.1 do 0.9.

1. Uruchom komórkę **TWOJA KOLEJ** bez zmian (kadr E: pas nawłoci, fraza `bush`, progi 0.1 / 0.3 / 0.5).
2. Zmień na `KADR = "B"`, `FRAZA = "dirt road"` i `PROGI = (0.5, 0.6, 0.8)` (ta sama droga i ta sama fałszywa plama co w zadaniu 2).


In [ ]:
# ==================== ZMIEŃ TU ====================
KADR = "E"
FRAZA = "bush"
PROGI = (0.1, 0.3, 0.5)      # trzy liczby z kropką, każda od 0.05 do 0.95
# ===================================================

zadanie_prog(KADR, FRAZA, PROGI)

---

## Krok 4. Plan B

### Zadanie 4. Gdy nazwa nie pomaga: ramka wzorcowa

Na kadrze A `tree` ma niską obecność i **nie znajduje nic**. Można jednak pokazać modelowi **przykład**: ramkę wokół jednego obiektu,
z poleceniem „szukaj czegoś takiego” (slajd 11). Ramka to cztery liczby, **piksele na osiach rysunku po lewej**:
`(x lewy, y górny, x prawy, y dolny)`. Okrągłe drzewo na kadrze A leży mniej więcej w ramce `(225, 115, 315, 225)`.

1. Uruchom komórkę **TWOJA KOLEJ** bez zmian. Ramka jest żółta, obok wynik bez ramki i z ramką.
2. Przesuń ramkę na **dużą kępę** po lewej: `(85, 25, 225, 165)`.
3. Wróć do pierwszej ramki i zmień `FRAZA` na `"tree crown"` (korona drzewa).


In [ ]:
# TWOJA KOLEJ
# ==================== ZMIEŃ TU ====================
KADR = "A"
FRAZA = "tree"                       # spróbuj też: "tree crown"
RAMKA = (225, 115, 315, 225)         # (x lewy, y górny, x prawy, y dolny) w pikselach kadru; duża kępa: (85, 25, 225, 165)
# ===================================================

zadanie_ramka(KADR, FRAZA, RAMKA)

---

## Krok 5. Całe zdjęcie

### Zadanie 5. Cała ortofotomapa i rozmiar kafla

Do tej pory pracowaliśmy na kadrach 40 × 40 m. Teraz to samo na **całym zdjęciu**: program tnie je na kafelki, na każdym szuka fraz,
skleja wyniki w **jedną maskę klas** (slajdy 12–15) i zapisuje dwa pliki dla QGIS-a: maskę (GeoTIFF) i poligony (GPKG).
Wybierasz: **jakie frazy** i **jaka klasa** do każdej (`roślinność`, `droga`, `dach`, `pojazd`) oraz **rozmiar kafla** w metrach.

Kafel to kompromis (slajd 12): **mały** = ostrzejsze brzegi, ale obiekt większy od kafla **znika** (maski dotykające krawędzi kafla są odrzucane).

1. Uruchom komórkę **TWOJA KOLEJ** bez zmian (`KAFEL_M = 60`; ok. 20 s na GPU). Zapisz w odpowiedziach, ile procent zdjęcia oznaczono.
2. Zmień `KAFEL_M` na `120` i uruchom jeszcze raz (szybciej). Porównaj obie maski i **dziennik przebiegów** pod rysunkiem.
3. Dopisz do słownika jedną własną frazę z zadań 1–4 (np. `"tire tracks": "droga"`) i uruchom jeszcze raz.


In [ ]:
# ==================== ZMIEŃ TU ====================
KAFEL_M = 60                                                       # rozmiar kafla w metrach: 60, potem 120
FRAZY_I_KLASY = {"bush": "roślinność", "dirt road": "droga"}       # "fraza": "klasa", pary oddzielone przecinkami
PROG_PEWNOSCI = 0.5                                                # próg pewności z zadania 3
# ===================================================

zadanie_cala(FRAZY_I_KLASY, KAFEL_M, PROG_PEWNOSCI)

---

---